# 03 — Sonar Pipeline: Classification, Detection, Segmentation

**Section A** — ResNet-50 classification (Watertank-Cropped, Turntable-Cropped)
**Section B** — YOLOv8m detection (Watertank-Segmentation bounding boxes)
**Section C** — SegFormer-B2 segmentation (Watertank-Segmentation pixel masks)

In [ ]:
import sys
sys.path.insert(0, '..')

import os
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from pathlib import Path
import matplotlib.pyplot as plt

from src.datasets import FLSClassificationDataset, FLSSegmentationDataset
from src.transforms import sonar_train_transforms, sonar_val_transforms, apply_sonar_transform
from src.train import FocalLoss, DiceLoss, CombinedLoss
from src.evaluate import evaluate_segmentation, plot_iou_per_class, plot_training_history, plot_confusion_matrix
from src.utils import get_device

DEVICE      = get_device()
DATA_ROOT   = Path('../marine-debris-fls-datasets/md_fls_dataset/data')
SEG_ROOT    = DATA_ROOT / 'watertank-segmentation'
RESULTS_DIR = Path('../results')
RESULTS_DIR.mkdir(exist_ok=True)

NUM_SEG_CLASSES = 12
SEG_CLASSES = [
    'background', 'bottle', 'can', 'chain', 'drink-carton',
    'hook', 'propeller', 'shampoo-bottle', 'standing-bottle', 'tire', 'valve', 'wall'
]

print('Device:', DEVICE)
print('Data root exists:', DATA_ROOT.exists())


## Section A — ResNet-50 Classification

Train on Watertank-Cropped (10 classes) and Turntable-Cropped (18 classes) separately.
Metrics: top-1 accuracy, per-class F1.

In [ ]:
import torchvision.models as tvm
from sklearn.metrics import classification_report, confusion_matrix
import numpy as np

def build_resnet50(num_classes: int, device: str):
    model = tvm.resnet50(weights=tvm.ResNet50_Weights.IMAGENET1K_V2)
    model.fc = nn.Linear(model.fc.in_features, num_classes)
    return model.to(device)

def train_classifier(root: str, split_name: str, epochs: int = 30,
                      batch_size: int = 32, lr: float = 3e-4) -> dict:
    train_ds = FLSClassificationDataset(root, split='train')
    val_ds   = FLSClassificationDataset(root, split='val')
    test_ds  = FLSClassificationDataset(root, split='test')
    num_classes = len(train_ds.classes)

    sampler = train_ds.make_weighted_sampler()
    train_loader = DataLoader(train_ds, batch_size=batch_size, sampler=sampler, num_workers=4)
    val_loader   = DataLoader(val_ds,   batch_size=batch_size, shuffle=False, num_workers=4)
    test_loader  = DataLoader(test_ds,  batch_size=batch_size, shuffle=False, num_workers=4)

    model     = build_resnet50(num_classes, DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs, eta_min=lr*0.01)
    criterion = nn.CrossEntropyLoss()  # standard CE for multi-class classification

    best_acc = 0.0
    checkpoint = str(RESULTS_DIR / f'cls_{split_name}_best.pt')
    history = []

    for epoch in range(1, epochs + 1):
        model.train()
        correct = total = 0
        for imgs, labels in train_loader:
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            optimizer.zero_grad()
            logits = model(imgs)          # single forward pass
            loss   = criterion(logits, labels)
            loss.backward()
            optimizer.step()
            correct += (logits.argmax(1) == labels).sum().item()
            total   += labels.size(0)
        train_acc = correct / total

        model.eval()
        correct = total = 0
        with torch.no_grad():
            for imgs, labels in val_loader:
                imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
                correct += (model(imgs).argmax(1) == labels).sum().item()
                total   += labels.size(0)
        val_acc = correct / total
        scheduler.step()
        history.append({'epoch': epoch, 'train_acc': train_acc, 'val_acc': val_acc})

        if val_acc > best_acc:
            best_acc = val_acc
            torch.save(model.state_dict(), checkpoint)
        if epoch % 5 == 0 or epoch == 1:
            print(f'Epoch {epoch:3d} | train {train_acc:.4f} | val {val_acc:.4f}')

    model.load_state_dict(torch.load(checkpoint, map_location=DEVICE))
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for imgs, labels in test_loader:
            all_preds.extend(model(imgs.to(DEVICE)).argmax(1).cpu().numpy())
            all_labels.extend(labels.numpy())

    report = classification_report(all_labels, all_preds, target_names=train_ds.classes,
                                    output_dict=True)
    print(f'\nTest accuracy ({split_name}): {report["accuracy"]:.4f}')
    print(classification_report(all_labels, all_preds, target_names=train_ds.classes))
    plot_confusion_matrix(np.array(all_preds), np.array(all_labels), train_ds.classes,
                          save_path=str(RESULTS_DIR / f'cls_{split_name}_confusion.png'))
    return {'history': history, 'test_acc': report["accuracy"], 'report': report,
            'checkpoint': checkpoint, 'classes': train_ds.classes}


### A1 — Watertank-Cropped (10 classes)

In [ ]:
watertank_results = train_classifier(
    root=str(DATA_ROOT / 'watertank-cropped'),
    split_name='watertank',
    epochs=30,
    batch_size=32,
)
# Plot training curve
epochs = [h['epoch'] for h in watertank_results['history']]
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(epochs, [h['train_acc'] for h in watertank_results['history']], label='train')
ax.plot(epochs, [h['val_acc']   for h in watertank_results['history']], label='val')
ax.set_title('ResNet-50 — Watertank Classification')
ax.set_xlabel('Epoch'); ax.set_ylabel('Accuracy'); ax.legend()
plt.tight_layout(); plt.savefig(RESULTS_DIR/'cls_watertank_curve.png', dpi=150); plt.show()


### A2 — Turntable-Cropped (18 classes)

> Note: class `rotating-platform` (186 images) is the turntable itself, not debris. Keep it in training for now — its F1 score in the report shows how easily the model can separate it.

In [ ]:
turntable_results = train_classifier(
    root=str(DATA_ROOT / 'turntable-cropped'),
    split_name='turntable',
    epochs=30,
    batch_size=32,
)
epochs = [h['epoch'] for h in turntable_results['history']]
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(epochs, [h['train_acc'] for h in turntable_results['history']], label='train')
ax.plot(epochs, [h['val_acc']   for h in turntable_results['history']], label='val')
ax.set_title('ResNet-50 — Turntable Classification')
ax.set_xlabel('Epoch'); ax.set_ylabel('Accuracy'); ax.legend()
plt.tight_layout(); plt.savefig(RESULTS_DIR/'cls_turntable_curve.png', dpi=150); plt.show()


## Section B — YOLOv8m Detection

Convert XML bounding boxes to YOLO format, then train YOLOv8m.
**Excludes wall (class 11) and background (class 0) from detection targets.**

In [ ]:
# ── One-time XML → YOLO label conversion ─────────────────────────────────
import xml.etree.ElementTree as ET
import shutil
from src.datasets import FLSDetectionDataset, _XML_NAME_MAP

YOLO_DIR = RESULTS_DIR / 'yolo_dataset'

# YOLO class mapping: exclude background(0) and wall(11); remap 1-10 → 0-9
DETECTION_CLASSES = [
    'bottle', 'can', 'chain', 'drink-carton', 'hook',
    'propeller', 'shampoo-bottle', 'standing-bottle', 'tire', 'valve'
]
DETECT_IDX_REMAP = {1: 0, 2: 1, 3: 2, 4: 3, 5: 4, 6: 5, 7: 6, 8: 7, 9: 8, 10: 9}

def convert_split(seg_root: Path, stems: list, split: str):
    img_out  = YOLO_DIR / split / 'images'
    lbl_out  = YOLO_DIR / split / 'labels'
    img_out.mkdir(parents=True, exist_ok=True)
    lbl_out.mkdir(parents=True, exist_ok=True)
    img_dir = seg_root / 'Images'
    ann_dir = seg_root / 'BoxAnnotations'
    skipped = 0
    for stem in stems:
        src_img = img_dir / (stem + '.png')
        src_xml = ann_dir / (stem + '.xml')
        if not src_img.exists() or not src_xml.exists():
            skipped += 1
            continue
        shutil.copy2(src_img, img_out / (stem + '.png'))

        tree = ET.parse(str(src_xml))
        root_elem = tree.getroot()
        size = root_elem.find('size')
        W = int(size.find('width').text)
        H = int(size.find('height').text)

        lines = []
        for obj in root_elem.findall('object'):
            name  = obj.find('name').text.strip()
            label = _XML_NAME_MAP.get(name, -1)
            if label < 0:
                label = next((v for k, v in _XML_NAME_MAP.items() if k.lower()==name.lower()), -1)
            if label <= 0 or label >= 11:  # skip background and wall
                continue
            yolo_cls = DETECT_IDX_REMAP[label]
            bb = obj.find('bndbox')
            x = float(bb.find('x').text)
            y = float(bb.find('y').text)
            w = float(bb.find('w').text)
            h = float(bb.find('h').text)
            cx = (x + w/2) / W
            cy = (y + h/2) / H
            nw = w / W
            nh = h / H
            cx, cy = max(0., min(1., cx)), max(0., min(1., cy))
            nw, nh = max(0., min(1., nw)), max(0., min(1., nh))
            if nw > 0 and nh > 0:
                lines.append(f'{yolo_cls} {cx:.6f} {cy:.6f} {nw:.6f} {nh:.6f}')
        with open(lbl_out / (stem + '.txt'), 'w') as f:
            f.write('\n'.join(lines))
    print(f'  {split}: {len(stems)-skipped} images, {skipped} skipped')

# Build dataset splits (reuse same seed as FLSDetectionDataset)
from sklearn.model_selection import train_test_split
all_stems = sorted(p.stem for p in (SEG_ROOT/'Images').glob('*.png')
                   if (SEG_ROOT/'BoxAnnotations'/(p.stem+'.xml')).exists())
tr_val, te = train_test_split(all_stems, test_size=0.15, random_state=42)
tr, va = train_test_split(tr_val, test_size=0.15/(1-0.15), random_state=42)

if not (YOLO_DIR / 'train' / 'images').exists():
    print('Converting XML → YOLO labels...')
    convert_split(SEG_ROOT, tr, 'train')
    convert_split(SEG_ROOT, va, 'val')
    convert_split(SEG_ROOT, te, 'test')
else:
    print('YOLO dataset already exists — skipping conversion')

# Write dataset.yaml
yaml_content = f'''path: {YOLO_DIR.resolve()}
train: train/images
val:   val/images
test:  test/images
nc: {len(DETECTION_CLASSES)}
names: {DETECTION_CLASSES}
'''
with open(YOLO_DIR / 'dataset.yaml', 'w') as f:
    f.write(yaml_content)
print('dataset.yaml written')


In [ ]:
# ── YOLOv8m training ─────────────────────────────────────────────────────
from ultralytics import YOLO

det_model = YOLO('yolov8m.pt')
det_results = det_model.train(
    data=str(YOLO_DIR / 'dataset.yaml'),
    epochs=80,
    imgsz=640,
    batch=8,
    device=DEVICE,
    project=str(RESULTS_DIR / 'yolo_sonar'),
    name='yolov8m_watertank',
    exist_ok=True,
    # disable colour augmentations — sonar is grayscale
    hsv_h=0.0,
    hsv_s=0.0,
    hsv_v=0.1,
)
print('Detection training complete.')
print(f'Best weights: {RESULTS_DIR}/yolo_sonar/yolov8m_watertank/weights/best.pt')


## Section C — SegFormer-B2 Segmentation

12 classes (0=background … 11=wall). mIoU computed on classes 1–10 only.
Background and wall excluded from loss and mIoU.

In [ ]:
import sys
sys.path.insert(0, '..')

import os
import numpy as np
import torch
from torch.utils.data import DataLoader
from transformers import SegformerForSemanticSegmentation
import matplotlib.pyplot as plt
from pathlib import Path

from src.datasets import FLSSegmentationDataset
from src.transforms import sonar_train_transforms, sonar_val_transforms, apply_sonar_transform
from src.train import run_training, CombinedLoss
from src.evaluate import evaluate_segmentation, plot_iou_per_class, plot_training_history
from src.utils import get_device, get_scaler, autocast_ctx

DEVICE      = get_device()
SEG_ROOT    = Path('../marine-debris-fls-datasets/md_fls_dataset/data/watertank-segmentation')
RESULTS_DIR = Path('../results')
RESULTS_DIR.mkdir(exist_ok=True)
CHECKPOINT  = str(RESULTS_DIR / 'sonar_best.pt')

NUM_CLASSES = 12  # 11 debris classes + background (class 0); wall=11 excluded from mIoU
FLS_CLASSES = [
    'background', 'bottle', 'can', 'chain', 'drink-carton',
    'hook', 'propeller', 'shampoo-bottle', 'standing-bottle', 'tire', 'valve', 'wall'
]

print('Device:', DEVICE)
print('Seg root exists:', SEG_ROOT.exists())


## Step 1 — Load Datasets

In [ ]:
train_aug = sonar_train_transforms(img_size=640)
val_aug   = sonar_val_transforms(img_size=640)

def make_transforms(aug):
    def fn(img, mask):
        return apply_sonar_transform(aug, img, mask)
    return fn

train_ds = FLSSegmentationDataset(str(SEG_ROOT), split='train',
                                   transform=make_transforms(train_aug))
val_ds   = FLSSegmentationDataset(str(SEG_ROOT), split='val',
                                   transform=make_transforms(val_aug))
test_ds  = FLSSegmentationDataset(str(SEG_ROOT), split='test',
                                   transform=make_transforms(val_aug))

train_sampler = train_ds.make_weighted_sampler()

train_loader = DataLoader(train_ds, batch_size=8, sampler=train_sampler,
                          num_workers=4, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=8, shuffle=False,
                          num_workers=4, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=8, shuffle=False,
                          num_workers=4, pin_memory=True)

print(f'Train: {len(train_ds)} | Val: {len(val_ds)} | Test: {len(test_ds)}')


## Step 2 — Build SegFormer-B2 Model

In [ ]:
# SegFormer-B2 pretrained on ADE20K, fine-tuned for FLS segmentation
model = SegformerForSemanticSegmentation.from_pretrained(
    'nvidia/segformer-b2-finetuned-ade-512-512',
    num_labels=NUM_CLASSES,
    ignore_mismatched_sizes=True,
)
model = model.to(DEVICE)

total_params = sum(p.numel() for p in model.parameters())
trainable    = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total params: {total_params/1e6:.1f}M | Trainable: {trainable/1e6:.1f}M')

## Step 3 — Custom Training Loop (Focal + Dice Loss)

SegFormer outputs logits at 1/4 resolution — we upsample before computing loss.

In [ ]:
import torch.nn.functional as F
from tqdm import tqdm
from src.train import FocalLoss, DiceLoss
from src.utils import get_scaler, autocast_ctx

focal_loss = FocalLoss(alpha=0.25, gamma=2.0)           # masks wall=11 by default
dice_loss  = DiceLoss(ignore_indices=(0, 11))           # skips background + wall

optimizer = torch.optim.AdamW(model.parameters(), lr=6e-5, weight_decay=1e-2)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=80, eta_min=1e-6)
scaler    = get_scaler(DEVICE)

EPOCHS = 80
best_miou = 0.0
history = []

for epoch in range(1, EPOCHS + 1):
    # ── Train ──
    model.train()
    train_loss = 0.0
    for imgs, masks in tqdm(train_loader, desc=f'Epoch {epoch}/{EPOCHS} [train]', leave=False):
        imgs, masks = imgs.to(DEVICE), masks.to(DEVICE)
        optimizer.zero_grad()

        if scaler:
            with autocast_ctx(DEVICE):
                out = model(pixel_values=imgs)
                logits = F.interpolate(out.logits, size=masks.shape[-2:],
                                       mode='bilinear', align_corners=False)
                loss = 0.5 * focal_loss(logits, masks) + 0.5 * dice_loss(logits, masks)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            out = model(pixel_values=imgs)
            logits = F.interpolate(out.logits, size=masks.shape[-2:],
                                   mode='bilinear', align_corners=False)
            loss = 0.5 * focal_loss(logits, masks) + 0.5 * dice_loss(logits, masks)
            loss.backward()
            optimizer.step()
        train_loss += loss.item()

    # ── Validate ──
    model.eval()
    val_loss = 0.0
    intersection = torch.zeros(NUM_CLASSES, device=DEVICE)
    union        = torch.zeros(NUM_CLASSES, device=DEVICE)

    with torch.no_grad():
        for imgs, masks in tqdm(val_loader, desc=f'Epoch {epoch}/{EPOCHS} [val]', leave=False):
            imgs, masks = imgs.to(DEVICE), masks.to(DEVICE)
            out = model(pixel_values=imgs)
            logits = F.interpolate(out.logits, size=masks.shape[-2:],
                                   mode='bilinear', align_corners=False)
            val_loss += (0.5 * focal_loss(logits, masks) + 0.5 * dice_loss(logits, masks)).item()
            preds = logits.argmax(dim=1)
            for cls in range(NUM_CLASSES):
                pred_c = preds == cls; tgt_c = masks == cls
                intersection[cls] += (pred_c & tgt_c).sum()
                union[cls]        += (pred_c | tgt_c).sum()

    iou_per_class = (intersection / (union + 1e-8)).cpu().numpy()
    # mIoU excludes background (0) and wall (11)
    valid = [i for i in range(1, NUM_CLASSES) if i != 11]
    miou = float(iou_per_class[valid].mean())
    scheduler.step()

    history.append({'epoch': epoch, 'train_loss': train_loss/len(train_loader),
                    'val_loss': val_loss/len(val_loader), 'miou': miou})
    print(f'Epoch {epoch:3d} | train {train_loss/len(train_loader):.4f} | '
          f'val {val_loss/len(val_loader):.4f} | mIoU {miou:.4f}')

    if miou > best_miou:
        best_miou = miou
        torch.save(model.state_dict(), CHECKPOINT)
        print(f'  -> Saved best model (mIoU={best_miou:.4f})')


## Step 4 — Evaluate on Test Set

In [ ]:
model.load_state_dict(torch.load(CHECKPOINT, map_location=DEVICE))
model.eval()

from src.evaluate import evaluate_segmentation

# Custom eval for SegFormer (handles 1/4-res logits)
all_preds, all_targets = [], []
with torch.no_grad():
    for imgs, masks in tqdm(test_loader, desc='Test eval'):
        imgs = imgs.to(DEVICE)
        out = model(pixel_values=imgs)
        logits = F.interpolate(out.logits, size=masks.shape[-2:],
                               mode='bilinear', align_corners=False)
        preds = logits.argmax(dim=1).cpu().numpy()
        all_preds.append(preds.flatten())
        all_targets.append(masks.numpy().flatten())

all_preds   = np.concatenate(all_preds)
all_targets = np.concatenate(all_targets)

from src.evaluate import compute_iou_per_class, plot_iou_per_class
iou = compute_iou_per_class(all_preds, all_targets, NUM_CLASSES)

# exclude background (0) and wall (11) from reported mIoU
valid = [i for i in range(1, NUM_CLASSES) if i != 11]
miou_test = float(iou[valid].mean())
pixel_acc = float((all_preds == all_targets).mean())

print(f'Test mIoU (excl. background + wall): {miou_test:.4f}')
print(f'Pixel accuracy: {pixel_acc:.4f}')
print('\nPer-class IoU:')
for cls, iou_val in zip(FLS_CLASSES, iou):
    print(f'  {cls:20s}: {iou_val:.4f}')


In [ ]:
plot_iou_per_class(iou, FLS_CLASSES, title='FLS Sonar — IoU per Class (Test)',
                   save_path=str(RESULTS_DIR / 'sonar_iou.png'))
plot_training_history(history, save_path=str(RESULTS_DIR / 'sonar_training.png'))

## Step 5 — Visualize Predictions

In [ ]:
import matplotlib.patches as mpatches

cmap = plt.colormaps['tab20'].resampled(NUM_CLASSES)

fig, axes = plt.subplots(4, 3, figsize=(12, 16))
model.eval()

sample_imgs, sample_masks = next(iter(test_loader))
with torch.no_grad():
    out = model(pixel_values=sample_imgs.to(DEVICE))
    logits = F.interpolate(out.logits, size=sample_masks.shape[-2:],
                           mode='bilinear', align_corners=False)
    pred_masks = logits.argmax(dim=1).cpu().numpy()

sample_imgs  = sample_imgs.numpy()
sample_masks = sample_masks.numpy()

for i, (ax_row) in enumerate(axes):
    if i >= len(sample_imgs):
        break
    img = sample_imgs[i].transpose(1, 2, 0)  # [H, W, 3]
    img = (img - img.min()) / (img.max() - img.min() + 1e-8)

    ax_row[0].imshow(img, cmap='gray')
    ax_row[0].set_title('Sonar Input')
    ax_row[0].axis('off')

    ax_row[1].imshow(sample_masks[i], cmap=cmap, vmin=0, vmax=NUM_CLASSES)
    ax_row[1].set_title('Ground Truth')
    ax_row[1].axis('off')

    ax_row[2].imshow(pred_masks[i], cmap=cmap, vmin=0, vmax=NUM_CLASSES)
    ax_row[2].set_title('Prediction')
    ax_row[2].axis('off')

plt.suptitle('SegFormer-B2 — Sonar Segmentation Predictions')
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'sonar_predictions.png', dpi=100)
plt.show()

## Results Summary

| Metric | Value |
|---|---|
| Test mIoU | *fill after run* |
| Pixel Accuracy | *fill after run* |

Checkpoint saved to: `results/sonar_best.pt` — used in `04_fusion.ipynb`.